# Travel time to the nearest station calcucation

In [1]:
import os

In [2]:
os.environ['PROJ_DATA'] = r'C:\Users\batur\micromamba\envs\geopython2025\Library\share\proj'

In [3]:
os.environ['PROJ_LIB'] = r'C:\Users\batur\micromamba\envs\geopython2025\Library\share\proj'

### 2. Cost distance calcucation
Goal: For every raster cell, we find the **minimum travel time** to the **nearest** station.

In [4]:
import numpy as np
import rasterio
import geopandas as gpd
from skimage.graph import MCP_Geometric
from rasterio.features import geometry_mask
from shapely.geometry import Point
from tqdm import tqdm


In [5]:
# Load raster
with rasterio.open("C:/PythonGIS/geopython2025/R_01/speed_m_min_SE_counties.tif") as src:
    speed = src.read(1)
    profile = src.profile
    transform = src.transform
    crs = src.crs


In [6]:
# Handle nodata/zero speed
speed = np.where((speed <= 0) | np.isnan(speed), np.nan, speed)

# Compute cost raster: minutes per meter
cost = 1.0 / speed
cost = np.clip(cost, 0.001, 1000)  # avoid division issues
cost[np.isnan(cost)] = 1000  # assign high cost to non-traversable cells


In [12]:
print(f"Raster shape: {speed.shape} — {speed.size / 1e6:.2f} million pixels")

Raster shape: (1940, 1695) — 3.29 million pixels


Speed raster have approx. 3.3 million pixels, so it might be take a time for cost distance analysis and later in R

In [7]:
# Load stations shapefile
stations = gpd.read_file("C:/PythonGIS/geopython2025/R_01/stations_counties_SE.shp").to_crs(crs)

In [8]:
# Function to convert to row/col
def coords_to_rowcol(geom, transform):
    x, y = geom.x, geom.y
    col, row = ~transform * (x, y)
    return int(row), int(col)

# Apply to all geometries
sources = [coords_to_rowcol(geom, transform) for geom in stations.geometry]

We use `MCP_Geometric` from `skimage.graph` — an implementation of **Dijkstra's algorithm** on a raster grid.

- **`fully_connected=True`** allows diagonal moves (8‑neighbour connectivity), which is more realistic for continuous travel.
- The algorithm computes the least‑cost path **from each cell to the nearest source**, not just between sources.
- We run it **once per station** and keep the **minimum** travel time across all stations.

#### Complexity
- Raster size: ~3.3 million cells
- 55 stations
- Total operations: ~180 million cell evaluations  
  (runs in ~4 minutes on a modern CPU)

#### Why this approach?
- **Exact** (not a heuristic like straight‑line distance)
- **Accounts for real barriers** (rivers, low‑speed roads)
- **Fully automatic** — no manual digitizing of catchment areas

In [9]:
# Prepare output array
travel_time_all = np.full(cost.shape, np.inf)

# Initialize MCP object once
mcp = MCP_Geometric(cost, fully_connected=True)

# Loop with progress bar
for i, source in enumerate(tqdm(sources, desc="Computing cost-distance")):
    try:
        travel_time, _ = mcp.find_costs(starts=[source])
        travel_time_all = np.minimum(travel_time_all, travel_time)
    except Exception as e:
        print(f"Station {i+1} failed: {e}")


Computing cost-distance: 100%|█████████████████████████████████████████████████████████| 55/55 [03:59<00:00,  4.35s/it]


In [10]:
out_profile = profile.copy()
out_profile.update(dtype='float32', count=1, nodata=np.nan)

# Replace inf with nodata
travel_time_all[~np.isfinite(travel_time_all)] = np.nan

# Save a raster
with rasterio.open("C:/PythonGIS/geopython2025/R_01/travel_time_SE.tif", "w", **out_profile) as dst:
    dst.write(travel_time_all.astype(np.float32), 1)


Calculated travel time was saved and analysis continued in Batura_fproj4.R